In [3]:
import pandas as pd

df = pd.read_csv('jigsaw-unintended-bias-in-toxicity-classification/all_data.csv')

In [10]:
len(df)

1999516

In [4]:
list(ds)

['comment_id',
 'annotator_id',
 'platform',
 'sentiment',
 'respect',
 'insult',
 'humiliate',
 'status',
 'dehumanize',
 'violence',
 'genocide',
 'attack_defend',
 'hatespeech',
 'hate_speech_score',
 'text',
 'infitms',
 'outfitms',
 'annotator_severity',
 'std_err',
 'annotator_infitms',
 'annotator_outfitms',
 'hypothesis',
 'target_race_asian',
 'target_race_black',
 'target_race_latinx',
 'target_race_middle_eastern',
 'target_race_native_american',
 'target_race_pacific_islander',
 'target_race_white',
 'target_race_other',
 'target_race',
 'target_religion_atheist',
 'target_religion_buddhist',
 'target_religion_christian',
 'target_religion_hindu',
 'target_religion_jewish',
 'target_religion_mormon',
 'target_religion_muslim',
 'target_religion_other',
 'target_religion',
 'target_origin_immigrant',
 'target_origin_migrant_worker',
 'target_origin_specific_country',
 'target_origin_undocumented',
 'target_origin_other',
 'target_origin',
 'target_gender_men',
 'target_gende

In [20]:
ds = ds[(ds.annotator_gender_men==True)|(ds.annotator_gender_women==True)]

In [28]:
ds['social_group'] = ds.apply(
    lambda row: 1 if row['gender'] == 0 and row['annotator_race_white'] == 1 else
                2 if row['gender'] == 1 and row['annotator_race_white'] == 1 else
                3 if row['gender'] == 0 and row['annotator_race_white'] == 0 else
                4,
    axis=1
)

### Compute different types of majority

In [60]:
grouped = relative_distribution.groupby('comment_id').agg({
    'violence': lambda x: list(zip(x, relative_distribution.loc[x.index, 'relative_distribution'])),
}).reset_index()

def determine_majority_type(group):
    if any(score == 1 for _, score in group):
        return 'unanimity'
    elif any(0.66 <= score < 1 for _, score in group):
        return 'absolute_majority'
    elif any(0.5 < score < 0.66 for _, score in group):
        return 'majority'
    else:
        return 'no_majority'

grouped['majority_type'] = grouped['violence'].apply(determine_majority_type)
grouped

,comment_id,violence,majority_type
0,1,"[(0.0, 0.5), (0.0, 0.25), (1.0, 0.25)]",no_majority
1,2,"[(0.0, 0.3333333333333333), (0.0, 0.3333333333...",no_majority
2,3,"[(0.0, 1.0)]",unanimity
3,4,"[(1.0, 1.0)]",unanimity
4,5,"[(0.0, 0.6666666666666666), (0.0, 0.3333333333...",absolute_majority
...,...,...,...
39456,50066,"[(1.0, 0.25), (3.0, 0.25), (3.0, 0.25), (4.0, ...",no_majority
39457,50067,"[(3.0, 1.0)]",unanimity
39458,50068,"[(0.0, 1.0)]",unanimity
39459,50069,"[(1.0, 1.0)]",unanimity


### compute the majority group for each comment

In [71]:
from collections import Counter
#violence = ds.groupby(['comment_id']).violence.apply(list).reset_index()
violence.violence = violence.violence.apply(lambda x: Counter(x).most_common(1)[0][0])

In [75]:
violence = violence.rename(columns={'violence': 'majority'})

violence = violence.merge(ds)

In [79]:
# Group by comment_id and majority, and count the occurrences of each social_group
grouped_data = violence.groupby(['comment_id', 'majority', 'social_group']).size().reset_index(name='count')

# Find the maximum count for each comment_id and majority
max_counts = grouped_data.groupby(['comment_id', 'majority'])['count'].max().reset_index(name='max_count')

# Merge the max_counts back to the grouped_data to filter the rows with the maximum count
merged_data = grouped_data.merge(max_counts, on=['comment_id', 'majority'])

# Filter rows where the count matches the max_count
filtered_data = merged_data[merged_data['count'] == merged_data['max_count']]

# Group by comment_id and majority, and collect social_groups with the highest count
result = filtered_data.groupby(['comment_id', 'majority'])['social_group'].apply(list).reset_index()

# Rename columns for clarity
result.rename(columns={'social_group': 'top_social_groups'}, inplace=True)

result

,comment_id,majority,top_social_groups
0,1,0.0,[1]
1,2,0.0,"[1, 2, 3]"
2,3,0.0,[1]
3,4,1.0,[1]
4,5,0.0,[2]
...,...,...,...
39456,50066,3.0,[1]
39457,50067,3.0,[2]
39458,50068,0.0,"[1, 2]"
39459,50069,1.0,[2]


In [86]:
res = result.top_social_groups.value_counts().reset_index()
res.to_csv('data/measuring_hatespeech/top_social_groups.csv', index=False)

### compute the number of time [0-1] in which each annotator diverge from the majority vote

In [95]:
anno = ds[['comment_id','annotator_id', 'social_group','violence']].drop_duplicates()

In [96]:
anno = anno.merge(result, on='comment_id')

In [106]:
# Calculate whether violence equals majority
anno['violence_equals_majority'] = anno['violence'] == anno['majority']

# Count the relative distribution of False in general
general_false_ratio = (~anno['violence_equals_majority']).mean()

# Count the relative distribution of False for each annotator
annotator_false_ratios = anno.groupby('annotator_id').apply(
    lambda group: (~group['violence_equals_majority']).mean()
).reset_index(name='false_ratio')

# Add the general false ratio as a new row for reference

annotator_false_ratios

/var/folders/f6/jg955hq50rn6mpdxyqdn_cb00000gn/T/ipykernel_18912/2932489752.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annotator_false_ratios = anno.groupby('annotator_id').apply(


,annotator_id,false_ratio
0,1,0.550000
1,3,0.533333
2,4,0.388889
3,5,0.222222
4,6,0.451613
...,...,...
7813,11138,0.272727
7814,11139,0.363636
7815,11140,0.320000
7816,11141,0.343750


In [115]:
l = list()

for item in anno.annotator_id.unique():
    length = len(anno[anno.annotator_id==item])
    ratio = anno[anno.annotator_id==item].violence_equals_majority.mean()
    l.append({
        'annotator_id': item,
        'false_ratio': ratio,
        'n_annotations': length,
    })

annotators = pd.DataFrame(l)

In [117]:
annotators.to_csv('data/measuring_hatespeech/annotator_false_ratios.csv', index=False)

In [1]:
import pandas as pd

df = pd.read_csv('jigsaw-unintended-bias-in-toxicity-classification/all_data.csv')

In [4]:
df = df.drop_duplicates(subset=['comment_text'])

In [6]:
df = df.sample(n=30000,random_state=42)



In [11]:
df.to_csv('jigsaw-unintended-bias-in-toxicity-classification/corpus_jigsaw.csv',index=False)

In [10]:
df.columns = ['id','text']

In [130]:
df = pd.read_csv('data/measuring_hatespeech/measuring - violence.csv')
grouped = df.groupby(['comment_id']).violence.apply(list).reset_index()
grouped['majority'] = grouped.violence.apply(lambda x: Counter(x).most_common(1)[0][0])
df = df.merge(grouped[['comment_id','majority']], on='comment_id')
df = df[df.violence == df.majority]

sg = df.groupby(['comment_id']).social_group.apply(list).reset_index()
print(sg)
sg['top_social_group'] = sg.social_group.apply(lambda x: Counter(x).most_common(1)[0][0]).reset_index(drop=True)
sg.top_social_group.value_counts()

       comment_id social_group
0               1    [1, 4, 2]
1               2       [1, 2]
2               3    [1, 1, 1]
3               4          [1]
4               5    [2, 2, 2]
...           ...          ...
39456       50066       [1, 2]
39457       50067          [2]
39458       50068       [2, 1]
39459       50069          [2]
39460       50070       [2, 2]

[39461 rows x 2 columns]


top_social_group
2    18612
1    13517
4     4208
3     3124
Name: count, dtype: int64

In [8]:
import pandas as pd

df = pd.read_csv('/Users/marco/Documents/aequa/calibration-bias/data/annWithAttitudes/largeScale.csv')



In [11]:
df

,Unnamed: 0,postId,tweet,ogId,ogLabel,source,ogLabelToxic,aae,hispanic,other,...,harmHateSpeech,intent,lingPurism,racism,racist,toany,toyou,traditionalism,off_avg,annId
0,0,545,Women are not fucking inferior to men,acl6143,nothate--none--none--original,dyna,False,0.195631,0.264603,0.118791,...,4,1,2,1,1,2,1,4,1.5,10638687
1,1,545,Women are not fucking inferior to men,acl6143,nothate--none--none--original,dyna,False,0.195631,0.264603,0.118791,...,4,1,4,1,1,1,1,4,1.0,8526465
2,2,3368,Elite Sub level RT @DanaeLovesYou: Putting a s...,t1090,normal--4,founta,False,0.379976,0.227157,0.056424,...,4,1,3,1,1,1,1,3,1.0,49021087
3,3,3368,Elite Sub level RT @DanaeLovesYou: Putting a s...,t1090,normal--4,founta,False,0.379976,0.227157,0.056424,...,5,1,4,1,1,1,1,5,1.0,10166032
4,4,1043,Karma is a bitch. Muslims will get what they d...,acl11558,hate--derogation--mus--perturbation,dyna,True,0.268704,0.285282,0.051341,...,4,5,2,1,5,5,5,4,5.0,10638687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3458,3458,1726,Well I'm glad that I haven't seen any of their...,acl34757,hate--derogation--bla--perturbation,dyna,True,0.094242,0.168711,0.006702,...,5,4,4,1,4,3,3,3,3.0,26767527
3459,3459,751,Why do they even offer Polish lessons in this ...,acl12881,hate--animosity--pol--perturbation,dyna,True,0.115153,0.289924,0.015129,...,3,4,4,2,4,4,4,4,4.0,27189963
3460,3460,751,Why do they even offer Polish lessons in this ...,acl12881,hate--animosity--pol--perturbation,dyna,True,0.115153,0.289924,0.015129,...,3,5,5,2,4,5,5,4,5.0,95971160
3461,3461,305,"I knew of ""burn a fag"", any time I've heard of...",acl26410,nothate--none--none--original,dyna,False,0.123040,0.283748,0.081284,...,5,5,5,1,1,5,5,3,5.0,15040445


In [14]:
df.to_csv('/Users/marco/Documents/aequa/calibration-bias/data/annWithAttitudes/corpus_attitudes.csv',index=False)